# 02 - Data Cleaning
Duplicate removal, missing-value imputation, outlier treatment (IQR & Z-score), datetime parsing, and validation.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
import pandas as pd
from src import config
from src.data_loader import load_raw_data
from src.preprocessing import (remove_duplicates, handle_missing_values, handle_outliers,
                                parse_datetime, detect_outliers_iqr, detect_outliers_zscore,
                                validate_dataset, clean_pipeline)

raw = load_raw_data()
print('Raw shape:', raw.shape)

2026-07-03 05:37:07 | src.data_loader | INFO | Loaded raw data: 12040 rows x 41 columns


2026-07-03 05:37:07 | src.data_loader | INFO | load_raw_data finished in 0.17s


Raw shape: (12040, 41)


## Step 1: Remove duplicates

In [3]:
df = remove_duplicates(raw)

2026-07-03 05:37:07 | src.preprocessing | INFO | Removed 40 duplicate rows (12040 -> 12000)


2026-07-03 05:37:07 | src.preprocessing | INFO | remove_duplicates finished in 0.03s


## Step 2: Parse datetime

In [4]:
df = parse_datetime(df)
df[config.DATETIME_COL].head()

2026-07-03 05:37:07 | src.preprocessing | INFO | parse_datetime finished in 0.03s


0   2024-06-01
1   2024-06-02
2   2024-06-03
3   2024-06-04
4   2024-06-05
Name: last_updated, dtype: datetime64[us]

## Step 3: Missing value imputation (median strategy)

In [5]:
print('Missing before:', df.isna().sum().sum())
df = handle_missing_values(df, strategy='median')
print('Missing after:', df.isna().sum().sum())

2026-07-03 05:37:07 | src.preprocessing | INFO | Missing-value imputation complete. Remaining NaNs: 0


2026-07-03 05:37:07 | src.preprocessing | INFO | handle_missing_values finished in 0.01s


Missing before: 1680
Missing after: 0


## Step 4: Outlier detection (IQR method) — demonstration on temperature

In [6]:
mask_iqr = detect_outliers_iqr(df['temperature_celsius'])
mask_z = detect_outliers_zscore(df['temperature_celsius'])
print(f'IQR outliers: {mask_iqr.sum()}')
print(f'Z-score outliers: {mask_z.sum()}')

IQR outliers: 37
Z-score outliers: 34


## Step 5: Outlier treatment (clip to IQR bounds) across numeric columns

In [7]:
df = handle_outliers(df, method='iqr', action='clip')

2026-07-03 05:37:08 | src.preprocessing | INFO | Outlier handling (iqr/clip): 6244 values flagged


2026-07-03 05:37:08 | src.preprocessing | INFO | handle_outliers finished in 0.14s


## Step 6: Validation report

In [8]:
report = validate_dataset(df)
report

2026-07-03 05:37:08 | src.preprocessing | INFO | Validation report: {'n_rows': 12000, 'n_duplicates': 0, 'n_missing_total': 0, 'negative_humidity': 0, 'humidity_over_100': 0, 'invalid_latitude': 0, 'invalid_longitude': 0}


{'n_rows': 12000,
 'n_duplicates': 0,
 'n_missing_total': 0,
 'negative_humidity': 0,
 'humidity_over_100': 0,
 'invalid_latitude': 0,
 'invalid_longitude': 0}

## Save cleaned dataset

In [9]:
df.to_csv(config.CLEAN_DATA_FILE, index=False)
print('Saved to', config.CLEAN_DATA_FILE)
print('Final shape:', df.shape)

Saved to /home/claude/Weather-Trend-Forecasting/data/processed/weather_clean.csv
Final shape: (12000, 41)
